# LpWM: dense state, sparse generator

This notebook runs the slot-free PushT or Wall experiment from `feature/sparse-generator`. It downloads the exact, unmodified DINO-WM datasets used by LpWM to Colab's local disk, while checkpoints, Hydra outputs, and W&B files are stored in MyDrive. The representation is a dense signed 8x8 patch field; exact top-k sparsity is used only for shared dynamics laws and token relations. Start with the smoke test before spending compute units.

In [ ]:
import os

if not os.path.exists('/content/lpworldmodel'):
    !git clone --branch feature/sparse-generator https://github.com/twojtys137/lpworldmodel.git /content/lpworldmodel
%cd /content/lpworldmodel
!git fetch origin feature/sparse-generator
!git checkout feature/sparse-generator
!git pull --ff-only origin feature/sparse-generator

In [ ]:
!pip -q install 'accelerate>=0.26,<2' 'hydra-core>=1.3,<2' 'omegaconf>=2.3,<3' 'wandb>=0.13,<1' einops decord pymunk pygame shapely scikit-image moviepy tensorboardX requests tqdm
!python -m pytest -q tests

## Exact LpWM data, MyDrive outputs, and W&B

Choose `pusht` (default) or `wall`. The downloader streams the corresponding original OSF archive to `/content/lpwm-data`, resumes interrupted downloads, verifies the official byte size and SHA-256, and extracts the original layout without conversion. PushT is 2.59 GiB compressed and Wall is 1.55 GiB compressed; the verified archive is removed after extraction. Local Colab storage is deliberately used for training speed.

In Colab, open **Secrets** (key icon), add a secret named `WANDB_API_KEY`, and grant this notebook access. Do not paste the key into a cell. The project `twojtys137-tw/lpwm-sparse-generator` is created automatically by the first authenticated run. If the secret is absent, the same notebook remains runnable in W&B offline mode and preserves its W&B files on MyDrive.

In [ ]:
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata
import wandb

drive.mount('/content/drive')

ENV_NAME = 'pusht'  # change to 'wall' for the original Wall dataset
DATASET_NAME = {'pusht': 'pusht_noise', 'wall': 'wall_single'}[ENV_NAME]
DATASET_DIR = Path('/content/lpwm-data')
RESULTS_DIR = Path('/content/drive/MyDrive/lpwm-sparse-generator')
WANDB_DIR = RESULTS_DIR / 'wandb'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
WANDB_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run([
    sys.executable, 'scripts/download_lpwmdatasets.py',
    '--dataset', DATASET_NAME, '--output-dir', str(DATASET_DIR),
], check=True)

os.environ.update({
    'DATASET_DIR': str(DATASET_DIR),
    'ENV_NAME': ENV_NAME,
    'CKPT_BASE': str(RESULTS_DIR),
    'WANDB_DIR': str(WANDB_DIR),
    'WANDB_ENTITY': 'twojtys137-tw',
    'WANDB_PROJECT': 'lpwm-sparse-generator',
})

try:
    wandb_key = userdata.get('WANDB_API_KEY')
except Exception:
    wandb_key = None
if wandb_key:
    wandb.login(key=wandb_key, relogin=True)
    os.environ['WANDB_MODE'] = 'online'
    print('W&B online: https://wandb.ai/twojtys137-tw/lpwm-sparse-generator')
else:
    os.environ['WANDB_MODE'] = 'offline'
    print('W&B offline: add the WANDB_API_KEY Colab secret to enable upload.')

print(f'Dataset: {DATASET_DIR / DATASET_NAME}')
print(f'Persistent results: {RESULTS_DIR}')

## 1. Smoke run

Eight rollouts, one epoch, batch 4 and 64 RDMReg projections. This validates data loading, gradients, checkpointing, and GPU memory.

In [ ]:
!SMOKE=1 bash scripts/train_sparse_generator_colab.sh

## 2. Screening run

The default is 50 rollouts, two epochs, batch 16, 64 patches, D=192, M=8, law top-k=2 and edge top-k=8. Set the flag only after the smoke run succeeds.

In [ ]:
RUN_FULL = False
if RUN_FULL:
    !SEED=0 bash scripts/train_sparse_generator_colab.sh

## 3. Controlled ablations

Keep the encoder and dense identity-linked state fixed. Compare `ltv`, `sparse_ltv`, `dense_generator`, and `sparse_generator`; promote only the best two to seeds 1 and 2. The detailed experiment matrix and interpretation of routing diagnostics are in `docs/sparse_generator.md`.

In [ ]:
# Print the matched commands; add RUN=1 (and initially SMOKE=1) to execute:
!bash scripts/sweep_sparse_generator_colab.sh
# !RUN=1 SMOKE=1 bash scripts/sweep_sparse_generator_colab.sh

# Example minimal sparse-LTV control using the same matched Colab config:
# !PREDICTOR=sparse_ltv NUM_PROJECTIONS=256 N_ROLLOUT=50 \
#   RUN_NAME=sparse_ltv_seed0 bash scripts/train_sparse_generator_colab.sh